In [ ]:
import rasterio
import numpy as np

def invert_01_tiff(input_path, output_path):
    """Invert 0-1 raster values (1 - value) and preserve NoData"""
    try:
        with rasterio.open(input_path) as src:
            data = src.read(1).astype(np.float32)
            meta = src.meta.copy()
            nodata = src.nodata if src.nodata is not None else np.nan

            # Convert NoData to NaN
            data = np.where(np.isclose(data, nodata), np.nan, data)

            # Invert values: 1 - original
            inverted = 1 - data

            # Update metadata
            meta.update(dtype=np.float32, nodata=np.nan)

            # Write output raster
            with rasterio.open(output_path, 'w', **meta) as dst:
                dst.write(inverted, 1)

        return True, f"✅ Inversion completed! Output file: {output_path}"

    except Exception as e:
        return False, f"❌ Processing failed: {str(e)}"


if __name__ == "__main__":
    input_tif = r"C:\...\NeglectedEvents_MSM_251120.tif"
    output_tif = r"C:\...\NeglectedEvents_MSM_251120_inverted.tif"

    success, msg = invert_01_tiff(input_tif, output_tif)
    print(msg)

In [ ]:
import rasterio
import numpy as np

def multiply_without_resampling(base_tif, coeff_tif, output_tif):
    """Multiply two rasters pixel-by-pixel without resampling"""
    with rasterio.open(base_tif) as base_src, \
         rasterio.open(coeff_tif) as coeff_src:

        # Read base raster data
        base_data = base_src.read(1).astype(np.float32)
        base_nodata = base_src.nodata if base_src.nodata is not None else np.nan

        # Read coefficient raster data
        coeff_data = coeff_src.read(1).astype(np.float32)
        coeff_nodata = coeff_src.nodata if coeff_src.nodata is not None else np.nan
        
        # Set 0 values in coefficient raster to NaN
        coeff_data[coeff_data == 0] = np.nan
        
        # Initialize result array with NaN
        result = np.full_like(base_data, np.nan, dtype=np.float32)

        # Iterate over each pixel of base raster
        for i in range(base_src.height):
            for j in range(base_src.width):
                # Skip NoData pixels in base raster
                if np.isclose(base_data[i, j], base_nodata):
                    continue

                # Get geographic coordinates of current pixel
                x, y = base_src.xy(i, j)

                # Get corresponding row and column in coefficient raster
                row, col = coeff_src.index(x, y)

                # Check if indices are within bounds
                if 0 <= row < coeff_src.height and 0 <= col < coeff_src.width:
                    cval = coeff_data[row, col]
                    # Multiply if coefficient value is valid
                    if not np.isclose(cval, coeff_nodata) and not np.isnan(cval):
                        result[i, j] = base_data[i, j] * cval

        # Save output raster
        meta = base_src.meta.copy()
        meta.update(dtype=np.float32, nodata=np.nan)
        with rasterio.open(output_tif, 'w', **meta) as dst:
            dst.write(result, 1)

    return True, f"✅ Completed! Output: {output_tif}"

if __name__ == "__main__":
    base_tif = r"C:\...\NeglectedEvents_MSM_251120_inverted.tif"
    coeff_tif = r"E:\...\worldpop_2020_log_ccy_norm.tif"
    output_tif = r"C:\...\NeglectedEvents_MSM_251120_PendInverted.tif"

    success, msg = multiply_without_resampling(base_tif, coeff_tif, output_tif)
    print(msg)

In [ ]:
import rasterio
import numpy as np

def invert_normalize_tiff(input_path, output_path):
    """Invert and normalize raster values to 0-1 range while preserving NoData"""
    try:
        with rasterio.open(input_path) as src:
            # Read data and metadata
            data = src.read(1).astype(np.float32)
            meta = src.meta.copy()
            nodata = src.nodata if src.nodata is not None else np.nan

            # Calculate valid data range (exclude NoData)
            valid_data = data[~np.isnan(data)]
            if valid_data.size == 0:
                raise ValueError("No valid pixels in the input file, cannot perform inverse normalization")
            
            min_orig = np.min(valid_data)
            max_orig = np.max(valid_data)

            # Avoid division by zero
            if max_orig == min_orig:
                inverted_data = 1.0 - data
            else:
                # Core inverse normalization formula
                inverted_data = (max_orig - data) / (max_orig - min_orig)

            # Restore NoData regions
            inverted_data[np.isnan(data)] = np.nan

            # Update metadata
            meta.update(
                dtype=np.float32,
                nodata=np.nan
            )

            # Write output raster
            with rasterio.open(output_path, 'w', **meta) as dst:
                dst.write(inverted_data, 1)

        return True, f"✅ Inverse normalization completed! Output: {output_path}" \
                     f"\nOriginal range: [{min_orig:.4f}, {max_orig:.4f}]" \
                     f"\nNormalized range: [{np.min(inverted_data[~np.isnan(inverted_data)]):.4f}, {np.max(inverted_data[~np.isnan(inverted_data)]):.4f}]"

    except Exception as e:
        return False, f"❌ Processing error: {str(e)}"

if __name__ == "__main__":
    input_tif = r"C:\...\NeglectedEvents_MSM_251120_PendInverted.tif"
    output_tif = r"C:\...\NeglectedEvents_MSM_251120_Pop.tif"

    success, msg = invert_normalize_tiff(input_tif, output_tif)
    print(msg)